In [1]:
import FinanceDataReader as fdr
import pandas as pd

In [2]:
df_krx = fdr.StockListing('KRX')[["Code", "Name"]]
df_krx

,Code,Name
0,005930,삼성전자
1,000660,SK하이닉스
2,005935,삼성전자우
3,402340,SK스퀘어
4,005380,현대차
...,...,...
2874,448780,마이크로엔엑스
2875,296520,가이아코퍼레이션
2876,266170,레드우즈
2877,279060,이노벡스


In [3]:

try:
    from IPython.display import display
except ImportError:
    display = print


def get_krx_code(df_krx, user_input, interactive=True):
    """
    df_krx에서 사용자가 입력한 Code 또는 Name을 검색하여
    최종 선택된 종목의 Code를 반환하는 함수
    
    Parameters
    ----------
    df_krx : DataFrame
        fdr.StockListing('KRX')[["Code", "Name"]] 형태의 데이터프레임
        
    user_input : str or int
        사용자가 입력한 종목코드 또는 종목명
        
    interactive : bool
        검색 결과가 여러 개인 경우 사용자에게 선택을 받을지 여부
        
    Returns
    -------
    selected_code : str
        선택된 종목코드
    """

    # 원본 df 보호
    df = df_krx.copy()

    # Code는 반드시 6자리 문자열로 처리
    df["Code"] = df["Code"].astype(str).str.zfill(6)
    df["Name"] = df["Name"].astype(str).str.strip()

    user_value = str(user_input).strip()

    if user_value == "":
        print("입력값이 비어 있습니다.")
        return None

    # 숫자만 입력된 경우 Code로 우선 검색
    if user_value.isdigit():
        code_value = user_value.zfill(6)

        # 1순위: Code 완전일치
        result = df[df["Code"] == code_value]

        # 2순위: Code 앞부분 일치
        if result.empty:
            result = df[df["Code"].str.startswith(user_value)]

    else:
        # 1순위: Name 완전일치
        result = df[df["Name"] == user_value]

        # 2순위: Name 부분일치
        if result.empty:
            result = df[
                df["Name"].str.contains(user_value, case=False, na=False, regex=False)
            ]

    # 검색 결과 없음
    if result.empty:
        print(f"'{user_value}'에 해당하는 종목을 찾지 못했습니다.")
        return None

    result = result.reset_index(drop=True)

    # 검색 결과가 1개인 경우 바로 확정
    if len(result) == 1:
        selected_code = result.loc[0, "Code"]
        selected_name = result.loc[0, "Name"]

        print("선택된 종목")
        display(result[["Code", "Name"]])

        return selected_code

    # 검색 결과가 여러 개인 경우
    print(f"'{user_value}' 검색 결과가 {len(result)}개 있습니다.")
    result_view = result[["Code", "Name"]].copy()
    result_view.insert(0, "Select_No", range(len(result_view)))

    display(result_view)

    if not interactive:
        print("검색 결과가 여러 개입니다. interactive=True로 실행하거나 더 정확한 값을 입력하세요.")
        return None

    while True:
        choice = input("사용할 종목의 Select_No를 입력하세요: ").strip()

        if choice.isdigit():
            choice = int(choice)

            if 0 <= choice < len(result):
                selected_code = result.loc[choice, "Code"]
                selected_name = result.loc[choice, "Name"]

                print("선택된 종목")
                display(result.loc[[choice], ["Code", "Name"]])

                return selected_code

        print("잘못된 선택입니다. Select_No 값을 다시 입력하세요.")

In [4]:
#  사용자가  입력
user_value = input("종목코드 또는 종목명을 입력하세요: ")

stock_code = get_krx_code(df_krx, user_value)

print("다음 단계로 전달할 Code:", stock_code)

종목코드 또는 종목명을 입력하세요:  한미반도체


선택된 종목


,Code,Name
0,042700,한미반도체


다음 단계로 전달할 Code: 042700


In [5]:
stock_code

'042700'

In [6]:
type(stock_code)

str

In [7]:
if stock_code is not None:
    # 이후 실행 단계에 Code 전달
    target_code = stock_code

    print("최종 사용 Code:", target_code)

    # 예시
    # df_price = fdr.DataReader(target_code, "2024-01-01")

최종 사용 Code: 042700


In [8]:
import requests
import time
import random
import re
import html
from bs4 import BeautifulSoup, UnicodeDammit


def extract_int(text):
    """
    문자열에서 숫자만 추출
    예: '1,234' -> 1234
    """
    if text is None:
        return None

    value = re.sub(r"[^0-9]", "", str(text))

    if value == "":
        return None

    return int(value)


def clean_text(text):
    """
    HTML entity와 공백 정리
    """
    if text is None:
        return ""

    text = html.unescape(str(text))
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def decode_naver_html(response):
    """
    네이버 금융 HTML 인코딩을 안전하게 디코딩
    """

    raw = response.content

    dammit = UnicodeDammit(
        raw,
        known_definite_encodings=[
            "utf-8",
            "cp949",
            "euc-kr"
        ]
    )

    html_text = dammit.unicode_markup

    return html_text, dammit.original_encoding

In [9]:
def crawl_naver_board_df(stock_code, start_page=1, end_page=10, sleep_range=(0.5, 1.2)):
    """
    네이버 금융 종목토론방에서 게시글 제목과 추천수를 수집하여 DataFrame으로 반환

    Parameters
    ----------
    stock_code : str
        종목코드. 예: '005930'

    start_page : int
        시작 페이지

    end_page : int
        종료 페이지

    sleep_range : tuple
        페이지 요청 사이 랜덤 대기 시간

    Returns
    -------
    DataFrame
        Title, Recommend_Count 컬럼을 가진 데이터프레임
    """

    stock_code = str(stock_code).zfill(6)

    base_url = "https://finance.naver.com/item/board.naver"

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Referer": f"https://finance.naver.com/item/main.naver?code={stock_code}",
        "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    }

    session = requests.Session()

    result_list = []

    for page in range(start_page, end_page + 1):

        params = {
            "code": stock_code,
            "page": page
        }

        try:
            response = session.get(
                base_url,
                params=params,
                headers=headers,
                timeout=10
            )

            response.raise_for_status()

            html_text, detected_encoding = decode_naver_html(response)

            soup = BeautifulSoup(html_text, "html.parser")

            rows = soup.select("table.type2 tr")

            page_count = 0

            for row in rows:
                title_td = row.select_one("td.title")

                if title_td is None:
                    continue

                a_tag = title_td.select_one("a")

                if a_tag is None:
                    continue

                # title 속성 우선 사용
                title_text = a_tag.get("title", "")

                # title 속성이 비어 있으면 a 태그 내부 텍스트 사용
                if title_text == "":
                    title_text = a_tag.get_text(" ", strip=True)

                title_text = clean_text(title_text)

                if title_text == "":
                    continue

                # 추천수 추출
                recommend_tag = row.select_one("strong.tah.p10.red01")

                if recommend_tag is not None:
                    recommend_count = extract_int(recommend_tag.get_text(strip=True))
                else:
                    # fallback: 게시판 구조상 뒤에서 두 번째 td가 추천수
                    tds = row.find_all("td", recursive=False)

                    if len(tds) >= 6:
                        recommend_count = extract_int(tds[-2].get_text(strip=True))
                    else:
                        recommend_count = None

                result_list.append({
                    "Title": title_text,
                    "Recommend_Count": recommend_count
                })

                page_count += 1

            print(
                f"{page}페이지 수집 완료: {page_count}건 "
                f"/ 누적 {len(result_list)}건 "
                f"/ 감지 인코딩: {detected_encoding}"
            )

            time.sleep(random.uniform(*sleep_range))

        except Exception as e:
            print(f"{page}페이지 수집 중 오류 발생: {e}")
            continue

    df_board = pd.DataFrame(result_list)

    # 컬럼 순서 고정
    if not df_board.empty:
        df_board = df_board[["Title", "Recommend_Count"]]

        # 추천수가 없는 경우 0으로 처리하고 정수형 변환
        df_board["Recommend_Count"] = (
            df_board["Recommend_Count"]
            .fillna(0)
            .astype(int)
        )

    return df_board

In [10]:

df_board = crawl_naver_board_df(
    stock_code=stock_code,
    start_page=1,
    end_page=10
)

df_board

1페이지 수집 완료: 20건 / 누적 20건 / 감지 인코딩: utf-8
2페이지 수집 완료: 20건 / 누적 40건 / 감지 인코딩: utf-8
3페이지 수집 완료: 20건 / 누적 60건 / 감지 인코딩: utf-8
4페이지 수집 완료: 20건 / 누적 80건 / 감지 인코딩: utf-8
5페이지 수집 완료: 20건 / 누적 100건 / 감지 인코딩: utf-8
6페이지 수집 완료: 20건 / 누적 120건 / 감지 인코딩: utf-8
7페이지 수집 완료: 20건 / 누적 140건 / 감지 인코딩: utf-8
8페이지 수집 완료: 20건 / 누적 160건 / 감지 인코딩: utf-8
9페이지 수집 완료: 20건 / 누적 180건 / 감지 인코딩: utf-8
10페이지 수집 완료: 20건 / 누적 200건 / 감지 인코딩: utf-8


,Title,Recommend_Count
0,유가 갑자기 하락하고 미선 좋아지고 있다,2
1,오늘까지 내려가는거 버티느라 고생많았습니다 ㅠㅠ,3
2,한미 본더는 HBM에 필수가 아니다.,1
3,좋은 기회다,2
4,반도체 슈퍼사이클과 지수급등으로 수배가 올랐는데,1
...,...,...
195,여기 고양이 프로필은,1
196,한화랑 특허 소송중인 상황 아시는분있나요?,0
197,삼천당 처음 폭락랄때 게시판보는것같네,1
198,목표가 40만?영업이익이 잘못된거같아,2


In [11]:
# !pip install transformers torch huggingface_hub -q

In [12]:
# from huggingface_hub import snapshot_download

# model_name = "snunlp/KR-FinBert-SC"
# local_model_path = "./models/KR-FinBert-SC"

# snapshot_download(
#     repo_id=model_name,
#     local_dir=local_model_path,
#     local_dir_use_symlinks=False
# )

# print("모델 다운로드 완료:", local_model_path)

In [13]:
import numpy as np
import torch
from transformers import pipeline


def analyze_board_buy_sell_signal(
    df_board,
    local_model_path="./models/KR-FinBert-SC",
    title_col="Title",
    recommend_col="Recommend_Count",
    batch_size=32,
    top_n=5
):
    """
    종목토론방 제목 데이터를 감정분석하여
    '매수' 또는 '매도' 의견과 근거 게시글 TOP N개를 반환한다.

    중립 의견은 사용자에게 노출하지 않는다.
    추천수는 weight로 반영한다.

    Parameters
    ----------
    df_board : DataFrame
        Title, Recommend_Count 컬럼을 가진 데이터프레임

    local_model_path : str
        로컬에 저장된 감정분석 모델 경로

    title_col : str
        게시글 제목 컬럼명

    recommend_col : str
        추천수 컬럼명

    batch_size : int
        모델 추론 batch size

    top_n : int
        근거 의견으로 노출할 게시글 개수

    Returns
    -------
    result : dict
        최종 판단과 근거 의견

    df_scored : DataFrame
        제목별 감정분석 점수 데이터
    """

    df = df_board.copy()

    # 기본 정리
    df[title_col] = df[title_col].astype(str).str.strip()
    df[recommend_col] = df[recommend_col].fillna(0).astype(int)

    # 빈 제목 제거
    df = df[df[title_col] != ""].reset_index(drop=True)

    if df.empty:
        raise ValueError("분석할 게시글 제목이 없습니다.")

    # 추천수 weight
    # 추천수가 큰 글일수록 영향력을 주되, 과도한 쏠림 방지를 위해 log scale 사용
    df["Weight"] = np.log1p(df[recommend_col]) + 1

    device = 0 if torch.cuda.is_available() else -1

    sentiment_pipe = pipeline(
        task="text-classification",
        model=local_model_path,
        tokenizer=local_model_path,
        device=device
    )

    texts = df[title_col].tolist()

    outputs = sentiment_pipe(
        texts,
        batch_size=batch_size,
        truncation=True,
        max_length=128,
        top_k=None
    )

    positive_probs = []
    negative_probs = []

    for result in outputs:
        score_dict = {
            item["label"].lower(): item["score"]
            for item in result
        }

        pos = score_dict.get("positive", 0)
        neg = score_dict.get("negative", 0)

        positive_probs.append(pos)
        negative_probs.append(neg)

    df["Prob_Positive"] = positive_probs
    df["Prob_Negative"] = negative_probs

    # 중립은 제외하고 positive vs negative만 비교
    df["Buy_Score"] = df["Weight"] * df["Prob_Positive"]
    df["Sell_Score"] = df["Weight"] * df["Prob_Negative"]

    total_buy_score = df["Buy_Score"].sum()
    total_sell_score = df["Sell_Score"].sum()

    if total_buy_score >= total_sell_score:
        final_signal = "매수"
        evidence_score_col = "Buy_Score"
        evidence_prob_col = "Prob_Positive"
    else:
        final_signal = "매도"
        evidence_score_col = "Sell_Score"
        evidence_prob_col = "Prob_Negative"

    # 최종 판단 방향에 부합하는 글만 근거 후보로 사용
    if final_signal == "매수":
        df_evidence = df[df["Buy_Score"] >= df["Sell_Score"]].copy()
    else:
        df_evidence = df[df["Sell_Score"] > df["Buy_Score"]].copy()

    # 근거 후보가 너무 적으면 전체에서 해당 방향 점수 높은 순으로 보완
    if df_evidence.empty:
        df_evidence = df.copy()

    df_evidence = df_evidence.sort_values(
        by=evidence_score_col,
        ascending=False
    ).head(top_n)

    evidence_texts = df_evidence[title_col].tolist()

    result = {
        "Signal": final_signal,
        "Buy_Score": round(float(total_buy_score), 4),
        "Sell_Score": round(float(total_sell_score), 4),
        "Evidence_Texts": evidence_texts
    }

    return result, df

In [14]:
def print_buy_sell_result(result):
    """
    사용자에게 최종 매수/매도 판단과 근거 의견 TOP 5만 노출
    """

    print("=" * 80)
    print(f"판단: {result['Signal']}")
    print("=" * 80)
    print()
    print("근거 의견 TOP 5")
    print("-" * 80)

    for idx, text in enumerate(result["Evidence_Texts"], start=1):
        print(f"{idx}. {text}")

In [15]:
result, df_scored = analyze_board_buy_sell_signal(
    df_board=df_board,
    local_model_path="./models/KR-FinBert-SC",
    top_n=5
)

print_buy_sell_result(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

판단: 매도

근거 의견 TOP 5
--------------------------------------------------------------------------------
1. 만약 몇일내에 삼전노조이슈 해결되면
2. 어차피 내일 30만원 깨짐
3. 조롱과비난
4. 한미 못 올라가도록. 공매로 누르고 비난하고.
5. 시총 35조에 영업익 85억? 한미반도체 ‘실적쇼크’…


In [16]:
df_scored[
    [
        "Title",
        "Recommend_Count",
        "Weight",
        "Prob_Positive",
        "Prob_Negative",
        "Buy_Score",
        "Sell_Score"
    ]
].sort_values(
    by=["Buy_Score", "Sell_Score"],
    ascending=False
).head(20)

,Title,Recommend_Count,Weight,Prob_Positive,Prob_Negative,Buy_Score,Sell_Score
47,"한미반도체 4,5월 수출실적",9,3.302585,0.995989,0.003715,3.289338,0.012268
13,"한미반도체 적정주가...25,000원",7,3.079442,0.964500,0.035199,2.970121,0.108394
133,선방했다,5,2.791759,0.984326,0.003900,2.748001,0.010887
83,회장이 사재 30억으로 자사주 매입했다고 하지않음?,4,2.609438,0.983102,0.000465,2.565343,0.001212
62,2/4분기 예상실적 좋음,8,3.197225,0.666747,0.133090,2.131741,0.425520
63,실적좋은 애들은 반등도 시원시원하게 나오네,2,2.098612,0.999742,0.000144,2.098071,0.000302
122,"속)한미 목표주가 5만 하향, 제주 18만 상향",1,1.693147,0.998516,0.001409,1.690634,0.002386
78,다시 오르네요,4,2.609438,0.647484,0.013608,1.689569,0.035508
162,연기금 매수했네~,14,3.708050,0.447508,0.010226,1.659380,0.037917
4,반도체 슈퍼사이클과 지수급등으로 수배가 올랐는데,1,1.693147,0.969173,0.008997,1.640952,0.015233
